In [65]:
def build_prompt(prediction, top_features, template="default"):
    """
    prediction: float
    top_features: list of tuples (name, raw_value, shap_value)
    Returns a single formatted prompt string.
    """
    lines = [
        "You are an AI assistant for choreographers.",
        f"Predicted score: {prediction:.2f}",
        "Top feature impacts:"
    ]
    for name, val, shap in top_features:
        lines.append(f"- {name}: {val} (impact {shap:+.2f})")
    lines.append("Explain why this score was predicted and suggest one improvement.")
    return "\n".join(lines)

In [ ]:
import openai, os
openai.api_key = os.getenv("OPENAI_API_KEY")

# def call_llm(prompt, model_name="gpt-3.5-turbo", max_tokens=200, temperature=0.7):
#     resp = openai.ChatCompletion.create(
#         model=model_name,
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=max_tokens,
#         temperature=temperature
#     )
#     return resp.choices[0].message.content.strip()


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [78]:
def call_llm_gemini(
    prompt: str,
    model_name: str = "gemini-2.5-flash",
    output_tokens: int = 200,
    temperature: float = 0.7
) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant for choreographers."},
        {"role": "user",   "content": prompt}
    ]

    resp = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        # thinking_config={"thinking_budget": 0},
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0) # Disables thinking
        ),
        max_output_tokens=output_tokens
    )

    text = resp.choices[0].message.content
    if not text:
        raise RuntimeError(f"No content returned (finish_reason={resp.choices[0].finish_reason})")
    return text.strip()

In [79]:
resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant for choreographers."},
        {"role": "user",   "content": prompt}
    ],
    max_tokens=200,
    temperature=0.7
)

In [80]:
print("Full response object:\n", resp, "\n")
print("Choices list:\n", resp.choices, "\n")
print("First choice object:\n", resp.choices[0], "\n")
print("Finish reason:", resp.choices[0].finish_reason)
print("Raw content field:", repr(resp.choices[0].message.content))

Full response object:
 ChatCompletion(id='hPKEaPCdOMK7vdIP2ISg0As', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1753543300, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=0, prompt_tokens=97, total_tokens=296, completion_tokens_details=None, prompt_tokens_details=None)) 

Choices list:
 [Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))] 

First choice object:
 Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)) 

Finish reason: length
R

In [81]:
import json
inst = json.load(open("shap/results/regression/Rhythm/sample_instances/instance_0.json"))

In [71]:
raw_feats = inst["features"]
shap_vals = inst["shap_values"]
top3 = sorted(shap_vals.items(), key=lambda x: abs(x[1]), reverse=True)[:3]
top_features = [(name, raw_feats[name], val) for name, val in top3]

In [72]:
prompt = build_prompt(inst["prediction"], top_features)
feedback = call_llm_gemini(prompt)
print(prompt)
print("---\n", feedback)

NameError: name 'types' is not defined

In [31]:
import os
print("Gemini key:", os.getenv("GEMINI_API_KEY"))


Gemini key: None


In [ ]:
import openai, inspect
print(openai)                # shows you where it’s loading from
print(inspect.getsource(openai))

<module 'openai' from '/Users/razie/Downloads/Evaluations_on_Robotic_Choreographies/venv-ext/lib/python3.8/site-packages/openai/__init__.py'>
# File generated from our OpenAPI spec by Stainless. See CONTRIBUTING.md for details.

from __future__ import annotations

import os as _os
import typing as _t
from typing_extensions import override

from . import types
from ._types import NOT_GIVEN, Omit, NoneType, NotGiven, Transport, ProxiesTypes
from ._utils import file_from_path
from ._client import Client, OpenAI, Stream, Timeout, Transport, AsyncClient, AsyncOpenAI, AsyncStream, RequestOptions
from ._models import BaseModel
from ._version import __title__, __version__
from ._response import APIResponse as APIResponse, AsyncAPIResponse as AsyncAPIResponse
from ._constants import DEFAULT_TIMEOUT, DEFAULT_MAX_RETRIES, DEFAULT_CONNECTION_LIMITS
from ._exceptions import (
    APIError,
    OpenAIError,
    ConflictError,
    NotFoundError,
    APIStatusError,
    RateLimitError,
    APITimeoutE

In [ ]:
import openai
print(openai.__version__)   # should be 0.27.0 or later


1.97.1


In [ ]:
import openai, inspect
print(openai)
print(openai.__file__)


<module 'openai' from '/Users/razie/Downloads/Evaluations_on_Robotic_Choreographies/venv-ext/lib/python3.8/site-packages/openai/__init__.py'>
/Users/razie/Downloads/Evaluations_on_Robotic_Choreographies/venv-ext/lib/python3.8/site-packages/openai/__init__.py


In [ ]:
import openai
print([name for name in dir(openai) if "Chat" in name])

['ChatCompletion']


In [ ]:
import openai
print(openai.__version__)            # should be 1.0.0
print([n for n in dir(openai) if "Chat" in n])  
# expect to see ['ChatCompletion', ...]

1.97.1
['ChatCompletion']
